- **logging into Postgres**
    - **docker compose exec postgres psql -U postgresuser -d oneshop**

- **List the tables inside oneshop**
    - **\dt** 

# **Modeling Iceberg Tables in the Lakehouse**

In [1]:
# Import the SparkSession entry point to programmatically configure and initialize Spark
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    # Name the application for identification in the Spark UI and cluster resource manager
    .appName("LakehouseLab")
    # 1. Enable Apache Iceberg SQL extensions (adds custom syntax like MERGE INTO, CREATE OR REPLACE, time travel, etc.)
    .config(
        "spark.sql.extensions",
        "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions",
    )
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262,org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.9.2")
    # 2. Register a new Iceberg catalog named 'lake' using Spark's SparkCatalog implementation
    .config("spark.sql.catalog.lake", "org.apache.iceberg.spark.SparkCatalog")
    # 3. Set the catalog backend type to 'rest' (communicates with an Iceberg REST catalog server for table metadata)
    .config("spark.sql.catalog.lake.type", "rest")
    # 4. Specify the URI endpoint of the Iceberg REST catalog service container
    .config("spark.sql.catalog.lake.uri", "http://iceberg-rest:8181")
    # 5. Define the root S3 warehouse directory path where Iceberg data files and manifests will be stored
    .config("spark.sql.catalog.lake.warehouse", "s3://warehouse/")
    # 6. Configure the IO implementation class for reading/writing files via S3 APIs
    .config(
        "spark.sql.catalog.lake.io-impl",
        "org.apache.iceberg.aws.s3.S3FileIO",
    )
    # 7. Point to the S3-compatible object storage endpoint (e.g., SeaweedFS or MinIO)
    .config("spark.sql.catalog.lake.s3.endpoint", "http://seaweedfs:8333")
    # 8. Force path-style access for S3 requests (required for many local/custom S3-compatible stores instead of virtual-hosted style)
    .config("spark.sql.catalog.lake.s3.path-style-access", "true")
    # 9. Provide the S3 storage authentication credentials (Access Key ID)
    .config("spark.sql.catalog.lake.s3.access-key-id", "lakehouse")
    # 10. Provide the S3 storage authentication credentials (Secret Access Key)
    .config("spark.sql.catalog.lake.s3.secret-access-key", "lakehouse-secret")
    # --- CRITICAL: Iceberg catalog-level S3 region settings ---
    .config("spark.sql.catalog.lake.s3.region", "us-east-1")
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
    .config("spark.hadoop.fs.s3a.endpoint", "http://seaweedfs:8333")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    # -----------------------------------------------------------
    # 11. Set 'lake' as the default catalog so tables can be referenced directly without prefixing the catalog name
    .config("spark.sql.defaultCatalog", "lake")
    # Materialize or retrieve the configured SparkSession instance
    .getOrCreate()
)

# Reduce noisy Spark execution logs by setting the root logger level to WARN instead of INFO
spark.sparkContext.setLogLevel("WARN")

# Print the active PySpark runtime version to verify successful initialization
print("Spark:", spark.version)

:: loading settings :: url = jar:file:/usr/local/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
org.apache.iceberg#iceberg-spark-runtime-3.5_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-8107326c-56d3-42af-bcd1-11f8951feba9;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
	found org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.9.2 in central
:: resolution report :: resolve 149ms :: artifacts dl 6ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	org.apache.hadoop#hadoop-aws;3.3.4 from central in [default]
	org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.9.2 from central in [default]
	org.wildfly.openssl#wildfly-openssl;1.0.7.Final fr

Spark: 3.5.6


In [2]:
from IPython.core.magic import register_cell_magic

@register_cell_magic
def sql(line, cell):
    result = spark.sql(cell)
    return result.show(100, truncate=False)

In [3]:
spark.catalog.setCurrentCatalog('lake')

In [4]:
spark.catalog.listCatalogs()

[CatalogMetadata(name='lake', description=None),
 CatalogMetadata(name='spark_catalog', description=None)]

## **Creating Namespaces (Bronz, Silver, Gold)**

> In Iceberg, a namespace is a logical container that organizes tables into groups, similar to how folders organize files in a file system. Namespaces help maintain a clean, hierarchical structure in your data lake, making it easier to manage access controls and organize related tables.

In [5]:
%%sql

CREATE DATABASE IF NOT EXISTS bronze;

++
||
++
++



In [6]:
%%sql 

CREATE DATABASE IF NOT EXISTS silver;

++
||
++
++



In [7]:
%%sql 

CREATE DATABASE IF NOT EXISTS gold;

++
||
++
++



In [8]:
spark.catalog.listDatabases()

[Database(name='bronze', catalog='lake', description=None, locationUri='s3://warehouse/bronze'),
 Database(name='silver', catalog='lake', description=None, locationUri='s3://warehouse/silver'),
 Database(name='gold', catalog='lake', description=None, locationUri='s3://warehouse/gold')]

## **Modeling Bronze Tables**

In [9]:
%%sql

CREATE TABLE IF NOT EXISTS bronze.users (
    id BIGINT,
    first_name STRING,
    last_name STRING,
    email STRING,
    created_at TIMESTAMP,
    updated_at TIMESTAMP
)
USING iceberg
PARTITIONED BY (days(created_at))
TBLPROPERTIES (
    'format-version' = '2',
    'comment' = 'Dimension table for user information'
);

++
||
++
++



In [10]:
%%sql

CREATE TABLE IF NOT EXISTS bronze.items (
    id BIGINT,
    name STRING,
    category STRING,
    price DECIMAL(7,2),
    inventory INT,
    created_at TIMESTAMP,
    updated_at TIMESTAMP
)
USING iceberg
PARTITIONED BY (category)
TBLPROPERTIES (
    'format-version' = '2',
    'comment' = 'Dimension table for product items'
);

++
||
++
++



In [11]:
%%sql

CREATE TABLE IF NOT EXISTS bronze.purchases (
    id BIGINT,
    user_id BIGINT,
    item_id BIGINT,
    quantity INT,
    purchase_price DECIMAL(12,2),
    created_at TIMESTAMP,
    updated_at TIMESTAMP
)
USING iceberg
PARTITIONED BY (days(created_at))
TBLPROPERTIES (
    'format-version' = '2',
    'comment' = 'Fact table for purchase transactions'
);

++
||
++
++



In [12]:
%%sql

CREATE TABLE IF NOT EXISTS bronze.pageviews (
    user_id BIGINT,
    url STRING,
    channel STRING,
    received_at TIMESTAMP
)
USING iceberg
PARTITIONED BY (days(received_at))
TBLPROPERTIES (
    'format-version' = '2',
    'comment' = 'Fact table for purchase transactions'
);

++
||
++
++



## **Modeling Silver Tables**

In [13]:
%%sql

CREATE TABLE IF NOT EXISTS silver.users (
    id BIGINT,
    first_name STRING,
    last_name STRING,
    email STRING,
    created_at TIMESTAMP,
    updated_at TIMESTAMP,
    valid_email BOOLEAN,
    full_name STRING
)
USING iceberg
PARTITIONED BY (days(created_at))
TBLPROPERTIES (
    'format-version' = '2',
    'comment' = 'Validated dimension table for user information'
);

++
||
++
++



In [14]:
%%sql
    
CREATE TABLE IF NOT EXISTS silver.items (
    id BIGINT,
    name STRING,
    category STRING,
    price DECIMAL(7,2),
    inventory INT,
    created_at TIMESTAMP,
    updated_at TIMESTAMP
)
USING iceberg
PARTITIONED BY (category)
TBLPROPERTIES (
    'format-version' = '2',
    'comment' = 'Dimension table for product items'
);

++
||
++
++



In [58]:
%%sql

DROP TABLE IF EXISTS silver.purchases_enriched;

++
||
++
++



In [59]:
%%sql
    
CREATE TABLE IF NOT EXISTS silver.purchases_enriched (
    id BIGINT,
    user_id BIGINT,
    item_id BIGINT,
    quantity INT,
    purchase_price DECIMAL(12,2),
    total_price DECIMAL(14,2),         
    user_email STRING,                
    item_name STRING,
    item_category STRING,
    purchase_date DATE,
    purchase_hour INT,
    created_at TIMESTAMP,
    updated_at TIMESTAMP                  
)
USING iceberg
PARTITIONED BY (days(created_at))
TBLPROPERTIES (
    'format-version' = '2',
    'comment' = 'Validated and enriched fact table for purchase transactions'
);

++
||
++
++



In [16]:
%%sql
    
CREATE TABLE IF NOT EXISTS silver.pageviews_by_items (
    user_id BIGINT,
    item_id BIGINT,
    page STRING,
    item_name STRING,
    item_category STRING,
    channel STRING,
    received_at TIMESTAMP
)
USING iceberg
PARTITIONED BY (days(received_at))
TBLPROPERTIES (
    'format-version' = '2',
    'comment' = 'Fact table for purchase transactions'
);

++
||
++
++



In [17]:
spark.catalog.listDatabases()

[Database(name='bronze', catalog='lake', description=None, locationUri='s3://warehouse/bronze'),
 Database(name='silver', catalog='lake', description=None, locationUri='s3://warehouse/silver'),
 Database(name='gold', catalog='lake', description=None, locationUri='s3://warehouse/gold')]

In [18]:
for db in ['bronze', 'silver', 'gold']:
    print(spark.catalog.listTables(dbName=db))

[Table(name='items', catalog='lake', namespace=['bronze'], description='Dimension table for product items', tableType='MANAGED', isTemporary=False), Table(name='pageviews', catalog='lake', namespace=['bronze'], description='Fact table for purchase transactions', tableType='MANAGED', isTemporary=False), Table(name='purchases', catalog='lake', namespace=['bronze'], description='Fact table for purchase transactions', tableType='MANAGED', isTemporary=False), Table(name='users', catalog='lake', namespace=['bronze'], description='Dimension table for user information', tableType='MANAGED', isTemporary=False)]
[Table(name='items', catalog='lake', namespace=['silver'], description='Dimension table for product items', tableType='MANAGED', isTemporary=False), Table(name='pageviews_by_items', catalog='lake', namespace=['silver'], description='Fact table for purchase transactions', tableType='MANAGED', isTemporary=False), Table(name='purchases_enriched', catalog='lake', namespace=['silver'], descri

In [19]:
%%sql

SHOW DATABASES;

+---------+
|namespace|
+---------+
|bronze   |
|silver   |
|gold     |
+---------+



In [20]:
%%sql

SHOW TABLES IN bronze;

+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
|bronze   |items    |false      |
|bronze   |pageviews|false      |
|bronze   |purchases|false      |
|bronze   |users    |false      |
+---------+---------+-----------+



In [21]:
%%sql

SHOW TABLES IN silver;

+---------+------------------+-----------+
|namespace|tableName         |isTemporary|
+---------+------------------+-----------+
|silver   |items             |false      |
|silver   |pageviews_by_items|false      |
|silver   |purchases_enriched|false      |
|silver   |users             |false      |
+---------+------------------+-----------+



In [22]:
%%sql

DESCRIBE TABLE bronze.users;

+--------------+----------------+-------+
|col_name      |data_type       |comment|
+--------------+----------------+-------+
|id            |bigint          |NULL   |
|first_name    |string          |NULL   |
|last_name     |string          |NULL   |
|email         |string          |NULL   |
|created_at    |timestamp       |NULL   |
|updated_at    |timestamp       |NULL   |
|              |                |       |
|# Partitioning|                |       |
|Part 0        |days(created_at)|       |
+--------------+----------------+-------+



In [23]:
%%sql

DESCRIBE TABLE silver.users;

+--------------+----------------+-------+
|col_name      |data_type       |comment|
+--------------+----------------+-------+
|id            |bigint          |NULL   |
|first_name    |string          |NULL   |
|last_name     |string          |NULL   |
|email         |string          |NULL   |
|created_at    |timestamp       |NULL   |
|updated_at    |timestamp       |NULL   |
|valid_email   |boolean         |NULL   |
|full_name     |string          |NULL   |
|              |                |       |
|# Partitioning|                |       |
|Part 0        |days(created_at)|       |
+--------------+----------------+-------+



In [24]:
%%sql

DESCRIBE TABLE EXTENDED bronze.users;

+----------------------------+----------------------------------------------------------------------------------------------------------------------+-------+
|col_name                    |data_type                                                                                                             |comment|
+----------------------------+----------------------------------------------------------------------------------------------------------------------+-------+
|id                          |bigint                                                                                                                |NULL   |
|first_name                  |string                                                                                                                |NULL   |
|last_name                   |string                                                                                                                |NULL   |
|email                       |string                

# **Loading Iceberg Tables from Postgres and SeaweedFS**

## **Loading Postgres Tables**

In [25]:
import sys
import pyspark.sql.functions as F

In [26]:
POSTGRES_URL = "jdbc:postgresql://postgres:5432/oneshop"
USERNAME = "postgresuser"
PASSWORD = "postgrespw"

In [27]:
# 1. Load users table
print("Processing 'users' table...")
try:
    users_df = spark.read \
        .format("jdbc") \
        .option("driver", "org.postgresql.Driver") \
        .option("url", POSTGRES_URL) \
        .option("dbtable", "users") \
        .option("user", USERNAME) \
        .option("password", PASSWORD) \
        .load()

    # Ensure column types match Iceberg schema (optional, but good practice)
    # Cast necessary columns if there are discrepancies, e.g., if Postgres SERIAL was read as INT
    users_df = users_df.select(
        F.col("id").cast("long"),
        F.col("first_name").cast("string"),
        F.col("last_name").cast("string"),
        F.col("email").cast("string"),
        F.current_timestamp().alias("created_at"),  # Generated dynamically if absent in source
        F.current_timestamp().alias("updated_at")   # Generated dynamically if absent in source
    )

    # Write to Iceberg table. 'overwrite' mode is used for initial load.
    users_df.write \
        .format("iceberg") \
        .mode("overwrite") \
        .option("format-version", "2") \
        .save("bronze.users")
    print("'users' table loaded successfully into Iceberg.")
except Exception as e:
    print(f"Error loading 'users' table: {e}")

Processing 'users' table...


'users' table loaded successfully into Iceberg.


In [28]:
spark.read.format('iceberg').load('bronze.users').show(10, truncate=False)

+---+----------+---------+---------------------------------+--------------------------+--------------------------+
|id |first_name|last_name|email                            |created_at                |updated_at                |
+---+----------+---------+---------------------------------+--------------------------+--------------------------+
|1  |Bob       |Haugen   |Elva.Crayton@inea.gov            |2026-11-16 17:04:48.817332|2026-11-16 17:04:48.817332|
|2  |Christian |Cox      |D.Barkley@adipiscingduis.info    |2026-11-16 17:04:48.817332|2026-11-16 17:04:48.817332|
|3  |Jere      |Crouch   |Edna@facilisinulla.com           |2026-11-16 17:04:48.817332|2026-11-16 17:04:48.817332|
|4  |Ethel     |Farmer   |Deon@lobortisautem.us            |2026-11-16 17:04:48.817332|2026-11-16 17:04:48.817332|
|5  |Iona      |Lay      |Theda@quifacilisis.gov           |2026-11-16 17:04:48.817332|2026-11-16 17:04:48.817332|
|6  |Len       |Watson   |Christoper.Herrin@delenitillum.tv|2026-11-16 17:04:48.

In [29]:
# 2. Load items table
print("Processing 'items' table...")
try:
    items_df = spark.read \
        .format("jdbc") \
        .option("url", POSTGRES_URL) \
        .option("dbtable", "public.items") \
        .option("user", USERNAME) \
        .option("password", PASSWORD) \
        .option("driver", "org.postgresql.Driver") \
        .load()

    # Cast necessary columns
    items_df = items_df.select(
        F.col("id").cast("long"),
        F.col("name").cast("string"),
        F.col("category").cast("string"),
        F.col("price").cast("decimal(7,2)"),
        F.col("inventory").cast("int"),
        F.current_timestamp().alias("created_at"),  # Generated dynamically if absent in source
        F.current_timestamp().alias("updated_at")   # Generated dynamically if absent in source
    )

    items_df.write \
        .format("iceberg") \
        .mode("overwrite") \
        .option("format-version", "2") \
        .save("bronze.items")
    print("'items' table loaded successfully into Iceberg.")
except Exception as e:
    print(f"Error loading 'items' table: {e}")

Processing 'items' table...
'items' table loaded successfully into Iceberg.


In [30]:
spark.read.format('iceberg').load('bronze.items').show(10, truncate=False)

+---+---------------------+---------+------+---------+--------------------------+--------------------------+
|id |name                 |category |price |inventory|created_at                |updated_at                |
+---+---------------------+---------+------+---------+--------------------------+--------------------------+
|1  |golf smile           |clearance|440.39|4576     |2026-11-16 17:04:50.913586|2026-11-16 17:04:50.913586|
|2  |pen stew             |clearance|7.04  |3961     |2026-11-16 17:04:50.913586|2026-11-16 17:04:50.913586|
|6  |Carol format         |clearance|181.34|3873     |2026-11-16 17:04:50.913586|2026-11-16 17:04:50.913586|
|12 |correspondent weather|clearance|307.52|3602     |2026-11-16 17:04:50.913586|2026-11-16 17:04:50.913586|
|15 |scent quicksand      |clearance|220.82|1803     |2026-11-16 17:04:50.913586|2026-11-16 17:04:50.913586|
|20 |underwear bagel      |clearance|78.59 |3231     |2026-11-16 17:04:50.913586|2026-11-16 17:04:50.913586|
|28 |cloudy year   

In [31]:
# 3. Load purchases table
print("Processing 'purchases' table...")
try:
    purchases_df = spark.read \
        .format("jdbc") \
        .option("url", POSTGRES_URL) \
        .option("dbtable", "public.purchases") \
        .option("user", USERNAME) \
        .option("password", PASSWORD) \
        .option("driver", "org.postgresql.Driver") \
        .load()

    # Cast necessary columns
    purchases_df = purchases_df.select(
        F.col("id").cast("long"),
        F.col("user_id").cast("long"),
        F.col("item_id").cast("long"),
        F.col("quantity").cast("int"),
        F.col("purchase_price").cast("decimal(12,2)"),
        F.current_timestamp().alias("created_at"),
        F.current_timestamp().alias("updated_at")
    )

    purchases_df.write \
        .format("iceberg") \
        .mode("overwrite") \
        .option("format-version", "2") \
        .save("bronze.purchases")
    print("'purchases' table loaded successfully into Iceberg.")
except Exception as e:
    print(f"Error loading 'purchases' table: {e}")

Processing 'purchases' table...
'purchases' table loaded successfully into Iceberg.


In [32]:
spark.read.format('iceberg').load('bronze.purchases').show(10, truncate=False)

+---+-------+-------+--------+--------------+--------------------------+--------------------------+
|id |user_id|item_id|quantity|purchase_price|created_at                |updated_at                |
+---+-------+-------+--------+--------------+--------------------------+--------------------------+
|1  |819    |191    |3       |254.52        |2026-11-16 17:04:51.426921|2026-11-16 17:04:51.426921|
|2  |6735   |450    |5       |536.15        |2026-11-16 17:04:51.426921|2026-11-16 17:04:51.426921|
|3  |386    |968    |5       |1454.55       |2026-11-16 17:04:51.426921|2026-11-16 17:04:51.426921|
|4  |8452   |982    |2       |839.58        |2026-11-16 17:04:51.426921|2026-11-16 17:04:51.426921|
|5  |2894   |193    |3       |728.04        |2026-11-16 17:04:51.426921|2026-11-16 17:04:51.426921|
|6  |3806   |512    |5       |1516.90       |2026-11-16 17:04:51.426921|2026-11-16 17:04:51.426921|
|7  |8343   |715    |3       |212.07        |2026-11-16 17:04:51.426921|2026-11-16 17:04:51.426921|


## **Loading PageView Events from SeaweedFS**

In [33]:
minio_endpoint = "http://seaweedfs:8333"

try:
    print("Reading pageview events from MinIO...")
    pageviews_df = spark.read.json("s3a://pageviews/")

    pageviews_df = pageviews_df.select(
        F.col("user_id").cast("long"),
        F.col("url").cast("string"),
        F.col("channel").cast("string"),
        F.col("received_at").cast("timestamp")
    )

    # Write to Iceberg table. 'overwrite' mode is used for initial load.
    pageviews_df.write \
        .format("iceberg") \
        .mode("overwrite") \
        .option("format-version", "2") \
        .save("bronze.pageviews")
    
    print("'pageviews' bucket content loaded successfully into Iceberg.")
except Exception as e:
    print(f"Error processing pageview events: {e}")

Reading pageview events from MinIO...


26/11/16 17:04:51 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
[Stage 34:=================================================>   (246 + 19) / 265]

'pageviews' bucket content loaded successfully into Iceberg.


In [34]:
spark.read.format('iceberg').load('bronze.pageviews').show(10, truncate=False)

+-------+--------------+--------------+-------------------+
|user_id|url           |channel       |received_at        |
+-------+--------------+--------------+-------------------+
|8959   |/products/1000|organic search|2026-11-16 15:18:45|
|8577   |/products/1000|organic search|2026-11-16 15:18:58|
|8168   |/products/586 |organic search|2026-11-16 15:18:38|
|5847   |/products/897 |organic search|2026-11-16 15:18:38|
|7545   |/products/132 |organic search|2026-11-16 15:18:39|
|9436   |/products/370 |organic search|2026-11-16 15:18:39|
|8944   |/products/915 |organic search|2026-11-16 15:18:39|
|2232   |/products/686 |organic search|2026-11-16 15:18:39|
|5067   |/products/230 |organic search|2026-11-16 15:18:39|
|9668   |/products/957 |organic search|2026-11-16 15:18:39|
+-------+--------------+--------------+-------------------+
only showing top 10 rows



In [36]:
spark.read.format('iceberg').load('bronze.pageviews').count()

8452

# **Data Cleaning, Denormalization, and Enrichment with PySpark**

In [37]:
bronze_users = spark.table("bronze.users")
bronze_items = spark.table("bronze.items")
bronze_purchases = spark.table("bronze.purchases")
bronze_pageviews = spark.table("bronze.pageviews")

In [38]:
# Define a simple email validation regex (basic, can be improved)
email_regex = r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$"

## **Transformations**

In [39]:
silver_users = (
    bronze_users
    .withColumn("valid_email", F.col("email").rlike(email_regex))
    .withColumn("full_name", F.concat_ws(" ", F.col("first_name"), F.col("last_name")))
)

In [41]:
silver_items = (
    bronze_items
    .withColumn("price", 
        F.when(F.col("price") < 0, F.lit(0)).otherwise(F.col("price"))
    )
    .withColumn("category", F.upper(F.col("category")))
)

In [56]:
silver_purchases = (
    bronze_purchases
    .join(bronze_users, bronze_purchases.user_id == bronze_users.id, "left")
    .join(bronze_items, bronze_purchases.item_id == bronze_items.id, "left")
    .select(
        bronze_purchases.id,
        bronze_purchases.user_id,
        bronze_purchases.item_id,
        bronze_purchases.quantity,
        bronze_purchases.purchase_price,
        (F.col("quantity") * F.col("purchase_price")).alias("total_price"),
        bronze_users.email.alias("user_email"),
        bronze_items.name.alias("item_name"),
        bronze_items.category.alias("item_category"),
        F.to_date(bronze_purchases.created_at).alias("purchase_date"),
        F.hour(bronze_purchases.created_at).alias("purchase_hour"),
        bronze_purchases.created_at,
        bronze_purchases.updated_at
    )
)

In [45]:
pageviews_with_item = bronze_pageviews.withColumn(
    "page", F.regexp_extract(F.col("url"), r"^/([^/]+)/\d+$", 1)
).withColumn(
    "item_id", F.regexp_extract(F.col("url"), r"/(\d+)$", 1).cast("bigint")
).filter(F.col("item_id").isNotNull())

In [46]:
silver_pageviews_by_items = (
    pageviews_with_item
    .join(bronze_items, pageviews_with_item.item_id == bronze_items.id, "left")
    .select(
        pageviews_with_item.user_id,
        pageviews_with_item.item_id,
        pageviews_with_item.page,
        bronze_items.name.alias("item_name"),
        bronze_items.category.alias("item_category"),
        pageviews_with_item.channel,
        pageviews_with_item.received_at
    )
)

In [47]:
(
    silver_users
    .select(
        "id",
        "first_name",
        "last_name",
        "email",
        "created_at",
        "updated_at",
        "valid_email",
        "full_name"
    )
    .writeTo("silver.users")
    .overwritePartitions()
)

In [48]:
(
    silver_items
    .select(
        "id",
        "name",
        "category",
        "price",
        "inventory",
        "created_at",
        "updated_at"
    )
    .writeTo("silver.items")
    .overwritePartitions()
)

In [60]:
(
    silver_purchases
    .writeTo("silver.purchases_enriched")
    .overwritePartitions()
)

In [61]:
(
    silver_pageviews_by_items
    .writeTo("silver.pageviews_by_items")
    .overwritePartitions()
)

# **Verify Silver Tables**

In [62]:
%%sql

SHOW DATABASES;

+---------+
|namespace|
+---------+
|bronze   |
|silver   |
|gold     |
+---------+



In [63]:
%%sql

SHOW TABLES IN silver;

+---------+------------------+-----------+
|namespace|tableName         |isTemporary|
+---------+------------------+-----------+
|silver   |items             |false      |
|silver   |pageviews_by_items|false      |
|silver   |purchases_enriched|false      |
|silver   |users             |false      |
+---------+------------------+-----------+



In [64]:
%%sql

SELECT * FROM silver.users LIMIT 10;

+---+----------+---------+---------------------------------+--------------------------+--------------------------+-----------+----------------+
|id |first_name|last_name|email                            |created_at                |updated_at                |valid_email|full_name       |
+---+----------+---------+---------------------------------+--------------------------+--------------------------+-----------+----------------+
|1  |Bob       |Haugen   |Elva.Crayton@inea.gov            |2026-11-16 17:04:48.817332|2026-11-16 17:04:48.817332|true       |Bob Haugen      |
|2  |Christian |Cox      |D.Barkley@adipiscingduis.info    |2026-11-16 17:04:48.817332|2026-11-16 17:04:48.817332|true       |Christian Cox   |
|3  |Jere      |Crouch   |Edna@facilisinulla.com           |2026-11-16 17:04:48.817332|2026-11-16 17:04:48.817332|true       |Jere Crouch     |
|4  |Ethel     |Farmer   |Deon@lobortisautem.us            |2026-11-16 17:04:48.817332|2026-11-16 17:04:48.817332|true       |Ethel Farm

In [66]:
%%sql

SELECT * FROM silver.items LIMIT 10;

+---+---------------------+---------+------+---------+--------------------------+--------------------------+
|id |name                 |category |price |inventory|created_at                |updated_at                |
+---+---------------------+---------+------+---------+--------------------------+--------------------------+
|1  |golf smile           |CLEARANCE|440.39|4576     |2026-11-16 17:04:50.913586|2026-11-16 17:04:50.913586|
|2  |pen stew             |CLEARANCE|7.04  |3961     |2026-11-16 17:04:50.913586|2026-11-16 17:04:50.913586|
|6  |Carol format         |CLEARANCE|181.34|3873     |2026-11-16 17:04:50.913586|2026-11-16 17:04:50.913586|
|12 |correspondent weather|CLEARANCE|307.52|3602     |2026-11-16 17:04:50.913586|2026-11-16 17:04:50.913586|
|15 |scent quicksand      |CLEARANCE|220.82|1803     |2026-11-16 17:04:50.913586|2026-11-16 17:04:50.913586|
|20 |underwear bagel      |CLEARANCE|78.59 |3231     |2026-11-16 17:04:50.913586|2026-11-16 17:04:50.913586|
|28 |cloudy year   

In [67]:
%%sql

SELECT * FROM silver.pageviews_by_items LIMIT 10;

+-------+-------+--------+------------------+-------------+--------------+-------------------+
|user_id|item_id|page    |item_name         |item_category|channel       |received_at        |
+-------+-------+--------+------------------+-------------+--------------+-------------------+
|8959   |1000   |products|cross textbook    |gadgets      |organic search|2026-11-16 15:18:45|
|8577   |1000   |products|cross textbook    |gadgets      |organic search|2026-11-16 15:18:58|
|8168   |586    |products|geometry protocol |gadgets      |organic search|2026-11-16 15:18:38|
|5847   |897    |products|instruction horse |clearance    |organic search|2026-11-16 15:18:38|
|7545   |132    |products|multimedia whiskey|gadgets      |organic search|2026-11-16 15:18:39|
|9436   |370    |products|summer soap       |doodads      |organic search|2026-11-16 15:18:39|
|8944   |915    |products|C-clamp water     |gadgets      |organic search|2026-11-16 15:18:39|
|2232   |686    |products|son offer         |clear

In [68]:
%%sql

SELECT * FROM silver.purchases_enriched LIMIT 10;

+---+-------+-------+--------+--------------+-----------+----------------------------------------+-------------------+-------------+-------------+-------------+--------------------------+--------------------------+
|id |user_id|item_id|quantity|purchase_price|total_price|user_email                              |item_name          |item_category|purchase_date|purchase_hour|created_at                |updated_at                |
+---+-------+-------+--------+--------------+-----------+----------------------------------------+-------------------+-------------+-------------+-------------+--------------------------+--------------------------+
|1  |819    |191    |3       |254.52        |763.56     |Lynn.Hutchison@inzzril.us               |comfort nickel     |widgets      |2026-11-16   |17           |2026-11-16 17:04:51.426921|2026-11-16 17:04:51.426921|
|2  |6735   |450    |5       |536.15        |2680.75    |Rayford.Serrano@facilisiqui.tv          |Persian sky        |clearance    |2026-11-

## **Using PyIceberg**

- **docker exec -it lakehouse-jupyter pyiceberg --catalog lake list**
- **docker exec -it lakehouse-jupyter pyiceberg --catalog lake list silver**
- **docker exec -it lakehouse-jupyter pyiceberg --catalog lake describe siver.table**